In [8]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus']=False
import pandas as pd
import numpy as np
from sklearn import svm,metrics
from sklearn.neighbors import KNeighborsClassifier,KNeighborsRegressor
from sklearn.linear_model import LinearRegression,Ridge,Lasso,LogisticRegression 
from sklearn.metrics import mean_absolute_error   # 평균절대값오차 오차범위
from sklearn.preprocessing import PolynomialFeatures,StandardScaler  
from sklearn.model_selection import train_test_split
from scipy.special import expit, softmax    # expit:이진분류,softmax:다중분류 0-1 숫자로 변환
from sklearn.linear_model import SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree
from sklearn.model_selection import cross_validate
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

In [9]:
# 로지스틱 회귀 - 다중분류 모델을 이용해서 생선을 분류하시오.
# 다중 : 소프트맥스 사용해서 퍼센트를 구하시오.
# 무게,길이,대각선,높이,넓이를 입력해서 생선을 분류해서 출력하시오.

In [10]:
fish = pd.read_csv('data/fish_logistic.csv')
fish.head(2)

,Species,Weight,Length,Diagonal,Height,Width
0,Bream,242.0,25.4,30.0,11.52,4.0200
1,Bream,290.0,26.3,31.2,12.48,4.3056


In [11]:
fish.info()

<class 'pandas.DataFrame'>
RangeIndex: 159 entries, 0 to 158
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Species   159 non-null    str    
 1   Weight    159 non-null    float64
 2   Length    159 non-null    float64
 3   Diagonal  159 non-null    float64
 4   Height    159 non-null    float64
 5   Width     159 non-null    float64
dtypes: float64(5), str(1)
memory usage: 7.6 KB


In [12]:
fish.columns

Index(['Species', 'Weight', 'Length', 'Diagonal', 'Height', 'Width'], dtype='str')

In [21]:
fish['Species'].unique()

<StringArray>
['Bream', 'Roach', 'Whitefish', 'Parkki', 'Perch', 'Pike', 'Smelt']
Length: 7, dtype: str

In [25]:
data = fish[['Weight', 'Length', 'Diagonal', 'Height', 'Width']].values
label = fish['Species'].values

In [26]:
train_data, test_data, train_label, test_label = train_test_split(data,label,test_size=0.2, random_state=42)

In [28]:
# 다중회귀 - 특성추가
poly = PolynomialFeatures(degree=5,include_bias=False)
poly.fit(train_data)
train_poly = poly.transform(train_data) #특성값추가
test_poly = poly.transform(test_data)

# 정규화작업
ss = StandardScaler()
ss.fit(train_poly)
train_scaled = ss.transform(train_poly)
test_scaled = ss.transform(test_poly)

In [30]:
lr = LogisticRegression(max_iter=1000)
lr.fit(train_scaled, train_label)

print('train 정확도 :', lr.score(train_scaled, train_label))
print('test 정확도 :', lr.score(test_scaled, test_label))

train 정확도 : 0.8740157480314961
test 정확도 : 0.90625


In [ ]:
# 테스트해볼 C 값들의 후보군 설정
params = {'C': [10, 50, 100, 200, 500, 1000]}

# GridSearchCV를 이용해 가장 성능이 좋은 최적의 모델 탐색
gs = GridSearchCV(LogisticRegression(max_iter=1000), params, cv=5)
gs.fit(train_scaled, train_label)

# 가장 이상적인 파라미터와 그때의 점수 확인
print("가장 이상적인 C 값:", gs.best_params_)
print("최고 교차 검증 점수:", gs.best_score_)

# 최적의 모델로 예측 및 저장
best_model = gs.best_estimator_

c:\python313\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(


가장 이상적인 C 값: {'C': 100}
최고 교차 검증 점수: 0.9446153846153846


In [32]:
# 1. 가장 이상적인 C 값인 100을 넣어 로지스틱 회귀 모델 생성
best_lr = LogisticRegression(C=100, max_iter=1000)

# 2. 정규화된 훈련 데이터(train_scaled)와 라벨(train_label)로 최종 학습
best_lr.fit(train_scaled, train_label)

# 3. 정확도 최종 확인
print('최종 train 정확도 :', best_lr.score(train_scaled, train_label))
print('최종 test 정확도 :', best_lr.score(test_scaled, test_label))

# ==========================================
# 4. 웹 연동을 위해 모델(.joblib) 저장하기
# ==========================================
import joblib

# 이미 찾아둔 최적의 모델을 model 변수에 담고
model = gs.best_estimator_

# 선생님이 말씀하신 이름으로 저장합니다!
joblib.dump(model, 'model.joblib2')
joblib.dump(ss, 'scaler.joblib') # 스케일러도 나중에 웹 연동할 때 필요하니 같이 저장해두면 좋습니다.

print('모델 저장 완료!')

최종 train 정확도 : 0.9921259842519685
최종 test 정확도 : 0.9375
모델 저장 완료!


In [33]:
import joblib
joblib.dump(ss, 'scaler.joblib')

['scaler.joblib']